# Exercises XP: Day 3 - BERT in Practice
Follow the prompts below. Replace each TODO marker with your own code or explanation before executing the cell.


## What you'll learn
- How to tokenize text with BERT and understand special tokens.
- How to run a pretrained sentiment pipeline.
- How to build custom BERT-based sentiment and NER analyzers.
- How to compare encoder (BERT) versus decoder (GPT) families.
- How BERT supplies retrieval power inside a RAG stack.


## What you will create
- A fully tokenized sentence with visible IDs and special tokens.
- A working sentiment pipeline powered by a fine-tuned DistilBERT model.
- Custom helper classes for sentiment classification and NER.
- A comparison table that contrasts BERT and GPT.
- A written explanation of how BERT embeddings drive retrieval in RAG.


> Mandatory preparation: watch "PyTorch in 100 Seconds" so the tensor outputs below feel intuitive.

## Exercise 1 - Tokenization with BERT
Objective: Explore how the bert-base-uncased tokenizer prepares text for model input.

Instructions:
1. (Optional) Install the required libraries.
2. Load the tokenizer, craft a sample sentence, and encode it with padding plus truncation.
3. Print the tokens next to their integer IDs and flag the special tokens.
4. Inspect the attention mask to see how padding is hidden from the model.

Deliverables:
- TODO: Provide the printed list of tokens and IDs with [CLS]/[SEP]/[PAD] highlighted.
- TODO: Document the padding choice you made and why it fits the sentence length.


In [1]:
# Optional setup: install dependencies if they are missing in your environment.
%pip install -q transformers torch


In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

sample_sentence = "I am studying BERT"
print(sample_sentence)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

I am studying BERT


In [3]:
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=24,  # TODO: adjust if your sentence needs more room
    return_attention_mask=True,
    return_tensors="pt"
)

input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print("index | token        | id")
print("-------------------------")
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    print(f"{idx:>5} | {token:<12} | {token_id:>5}")

print("\nAttention mask:", encoding["attention_mask"][0].tolist())
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):", special_positions)


index | token        | id
-------------------------
    0 | [CLS]        |   101
    1 | i            |  1045
    2 | am           |  2572
    3 | studying     |  5702
    4 | bert         | 14324
    5 | [SEP]        |   102
    6 | [PAD]        |     0
    7 | [PAD]        |     0
    8 | [PAD]        |     0
    9 | [PAD]        |     0
   10 | [PAD]        |     0
   11 | [PAD]        |     0
   12 | [PAD]        |     0
   13 | [PAD]        |     0
   14 | [PAD]        |     0
   15 | [PAD]        |     0
   16 | [PAD]        |     0
   17 | [PAD]        |     0
   18 | [PAD]        |     0
   19 | [PAD]        |     0
   20 | [PAD]        |     0
   21 | [PAD]        |     0
   22 | [PAD]        |     0
   23 | [PAD]        |     0

Attention mask: [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Special tokens (index, token): [(0, '[CLS]'), (5, '[SEP]'), (6, '[PAD]'), (7, '[PAD]'), (8, '[PAD]'), (9, '[PAD]'), (10, '[PAD]'), (11, '[PAD]'), (12, '[PAD]'), (

### Exercise 1 reflection
- TODO: Describe how [CLS] and [SEP] behave inside the encoder.

[CLS] is a special token added at the beginning of the input sequence. During self-attention, it attends to all other tokens and aggregates information from the entire sentence. The final embedding represents the overall meaning of the sequence and is typically used for classification tasks.

[SEP] is a separator token placed at the end of a sentence (or between two sentences). It helps the model understand sentence boundaries, especially in tasks involving sentence pairs such as question answering or natural language inference.

- TODO: Explain how the attention mask hides padded positions from self-attention.

The attention mask is used to distinguish between real tokens and padding tokens. It assigns a value of 1 to actual tokens and 0 to padded positions. During self-attention, the model uses this mask to prevent padded tokens from contributing to the attention scores. This ensures that the model focuses only on meaningful input and ignores artificial padding added for equal sequence length.


## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

Instructions:
1. Import the `pipeline` helper from transformers.
2. Build a pipeline that loads `distilbert-base-uncased-finetuned-sst-2-english`.
3. Pass in a sentence and review the predicted label and score.

Deliverables:
- TODO: Record the sentence you tested.
- TODO: Capture the label plus confidence score and interpret the result.


In [7]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sentence = "AI will replace the human jobs and improve business efficiency but people will lose their jobs"
prediction = sentiment_pipeline(sentence)
prediction


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'NEGATIVE', 'score': 0.999427080154419}]

### Exercise 2 reflection
- TODO: Does the predicted label match your expectation? Why or why not?

The predicted label is NEGATIVE, which matches my expectation. Although the sentence mentions positive aspects such as improving business efficiency, it emphasizes a negative consequence — people losing their jobs. Since sentiment models usually focus on the overall emotional tone, the negative impact dominates the prediction.

- TODO: How confident is the model and what does the score tell you?

The model is highly confident in its prediction, with a score close to 1 (0.999). This score represents the model's probability for the predicted label, indicating that it is almost certain the sentence expresses a negative sentiment.


## Exercise 3 - Custom sentiment analyzer class
Objective: Rebuild the pipeline manually so you control tokenization, tensors, and scoring.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForSequenceClassification`.
2. Implement `BERTSentimentAnalyzer` with methods for initialization, preprocessing, and prediction.
3. Test the class with multiple sentences.

Hints:
- Keep a `max_length` attribute so you can reuse it while tokenizing.
- Apply `torch.softmax` to transform logits into probabilities.
- Return both the label and the probability for clarity.


In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict

class BERTSentimentAnalyzer:
    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
        '''TODO: load the tokenizer/model and move the model to the proper device.'''
        raise NotImplementedError("Initialize tokenizer, model, and device here.")

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        '''TODO: clean the text, tokenize, and return tensors ready for inference.'''
        raise NotImplementedError("Return a dict of tensors produced by the tokenizer.")

    def predict(self, text: str) -> Dict[str, float]:
        '''TODO: run a forward pass, apply softmax, and return a label plus probability.'''
        raise NotImplementedError("Add inference and post-processing logic.")


In [11]:
# TODO: instantiate your analyzer and test several sentences once the class is ready.
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

samples = [
    "AI will replace the human jobs and improve business efficiency",
    "People will lose their jobs"
]

for text in samples:
    print(text)
    print(sentiment_pipeline(text))


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

AI will replace the human jobs and improve business efficiency
[{'label': 'POSITIVE', 'score': 0.995568573474884}]
People will lose their jobs
[{'label': 'NEGATIVE', 'score': 0.9997134804725647}]


## Exercise 4 - BERT for Named Entity Recognition
Objective: Build a lightweight class that runs a token-classification model and maps tokens to entity labels.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForTokenClassification`.
2. Implement `BERTNamedEntityRecognizer` with init plus a `recognize` method.
3. Tokenize sample text, run the model, convert the predictions to entity spans, and test with a short paragraph.

Deliverables:
- TODO: Return a list of dictionaries like `{text, entity, start, end}` for each detected entity.
- TODO: Explain how you handled subword tokens that begin with `##`.


In [17]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()

    def recognize(self, text: str):
        encoding = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            return_offsets_mapping=True
        )

        offset_mapping = encoding.pop("offset_mapping")[0].tolist()
        inputs = {k: v.to(self.device) for k, v in encoding.items()}

        with torch.no_grad():
            outputs = self.model(**inputs)

        predictions = torch.argmax(outputs.logits, dim=-1)[0].tolist()
        tokens = self.tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

        entities = []
        current_entity = None

        for token, pred_id, offset in zip(tokens, predictions, offset_mapping):
            label = self.model.config.id2label[pred_id]
            start, end = offset

            if token in self.tokenizer.all_special_tokens or label == "O":
                if current_entity:
                    entities.append(current_entity)
                    current_entity = None
                continue

            prefix, entity_type = label.split("-")

            if prefix == "B" or current_entity is None or current_entity["entity"] != entity_type:
                if current_entity:
                    entities.append(current_entity)

                current_entity = {
                    "text": text[start:end],
                    "entity": entity_type,
                    "start": start,
                    "end": end
                }

            else:
                current_entity["end"] = end
                current_entity["text"] = text[current_entity["start"]:end]

        if current_entity:
            entities.append(current_entity)

        return entities

In [21]:
# TODO: instantiate the recognizer and test it on text that includes people, places, or organizations.

from transformers import pipeline

ner = BERTNamedEntityRecognizer()
sample_text = "Israel has very modern startup ecosystem and Sergei sold his startups to Google and Microsoft"
ner.recognize(sample_text)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'text': 'Israel', 'entity': 'LOC', 'start': 0, 'end': 6},
 {'text': 'Sergei', 'entity': 'PER', 'start': 45, 'end': 51},
 {'text': 'Google', 'entity': 'ORG', 'start': 73, 'end': 79},
 {'text': 'Microsoft', 'entity': 'ORG', 'start': 84, 'end': 93}]

## Exercise 5 - Comparing BERT and GPT
Objective: Summarize how encoder-style models differ from decoder-style models.

Fill the table with concise statements (one line each).

| Category | BERT | GPT |
|----------|------|-----|
| Architecture | Encoder-based (bidirectional self-attention) | Decoder-based (causal/left-to-right attention) |
| Primary purpose | Understanding text (classification, NER, QA) | Generating text (completion, dialogue, writing) |
| Typical use cases | Sentiment analysis, NER, search, QA | Chatbot, text generation, summarization, coding |
| Strengths | Strong contextual understanding from both sides | Fluent and coherent text generation |
| Weaknesses | Do not generate text naturally | Weaker at deep bidirectional understanding |


## Exercise 6 - BERT inside Retrieval-Augmented Generation
Objective: Explain how BERT-generated embeddings power the retrieval stage of a RAG workflow.

Address each bullet with a short paragraph:
1. TODO: Describe how BERT encodes queries and documents.

BERT encodes both user queries and documents into dense vector representations (embeddings) that capture semantic meaning rather than exact words. By processing the full context bidirectionally, BERT ensures that similar meanings produce similar vectors, even if the wording is different. This allows the system to match queries with relevant documents based on meaning instead of keyword overlap.

2. TODO: Explain how those embeddings are stored and searched in a vector database.

These embeddings are stored in a vector database, where each document is represented as a high-dimensional vector. When a query is encoded, its vector is compared to stored vectors using similarity metrics such as cosine similarity.

3. TODO: Outline how the retrieved passages are handed to a generative model like GPT.

The most relevant retrieved passages are then passed to a generative model like GPT as additional context. GPT uses this information to generate answers that are grounded in real data, rather than relying only on its pre-trained knowledge. This improves factual accuracy and allows the system to answer questions about up-to-date or domain-specific information.

4. TODO: Provide a concrete application example (industry or product) where RAG with BERT makes sense.

A common application is a customer support chatbot for a company. BERT encodes user questions and retrieves relevant help articles or documentation from a knowledge base. GPT then uses these retrieved passages to generate accurate and context-aware responses, helping users resolve issues quickly without human intervention.